In [1]:
import requests

response = requests.get("https://api.tcgdex.net/v2/en/sets")
print(response.status_code)
print(response.json()[:3])

200
[{'id': 'miscp', 'name': 'Miscellaneous Promos', 'cardCount': {'total': 1, 'official': 1}}, {'id': 'base1', 'name': 'Base Set', 'logo': 'https://assets.tcgdex.net/en/base/base1/logo', 'cardCount': {'total': 102, 'official': 102}}, {'id': 'base2', 'name': 'Jungle', 'logo': 'https://assets.tcgdex.net/en/base/base2/logo', 'symbol': 'https://assets.tcgdex.net/univ/base/base2/symbol', 'cardCount': {'total': 64, 'official': 64}}]


In [2]:
response = requests.get("https://api.tcgdex.net/v2/en/sets")
sets = response.json()

for s in sets:
    if s['name'] in ['Base Set', 'Jungle', 'Fossil']:
        print(s['id'], s['name'], s['cardCount'])

base1 Base Set {'total': 102, 'official': 102}
base2 Jungle {'total': 64, 'official': 64}
base3 Fossil {'total': 62, 'official': 62}


In [3]:
response = requests.get("https://api.tcgdex.net/v2/en/sets/base1")
base1_data = response.json()

print(base1_data['cards'][:3])

[{'id': 'base1-1', 'image': 'https://assets.tcgdex.net/en/base/base1/1', 'localId': '1', 'name': 'Alakazam'}, {'id': 'base1-2', 'image': 'https://assets.tcgdex.net/en/base/base1/2', 'localId': '2', 'name': 'Blastoise'}, {'id': 'base1-3', 'image': 'https://assets.tcgdex.net/en/base/base1/3', 'localId': '3', 'name': 'Chansey'}]


In [4]:
response = requests.get("https://api.tcgdex.net/v2/en/cards/base1-1")
card_detail = response.json()

print(card_detail)

{'category': 'Pokemon', 'id': 'base1-1', 'illustrator': 'Ken Sugimori', 'image': 'https://assets.tcgdex.net/en/base/base1/1', 'localId': '1', 'name': 'Alakazam', 'rarity': 'Rare', 'set': {'cardCount': {'official': 102, 'total': 102}, 'id': 'base1', 'logo': 'https://assets.tcgdex.net/en/base/base1/logo', 'name': 'Base Set'}, 'variants': {'firstEdition': True, 'holo': True, 'normal': False, 'reverse': False, 'wPromo': False}, 'variants_detailed': [{'type': 'holo', 'subtype': 'unlimited', 'size': 'standard', 'thirdParty': {'cardmarket': 273696, 'tcgplayer': 42346}, 'variantId': '4ffrmhcfiaejakhepqdkx7o', 'pricing': {'cardmarket': {'updated': '2026-09-30T22:54:32.971Z', 'unit': 'EUR', 'idProduct': 273696, 'avg': 51.3, 'low': 9, 'trend': 48.78, 'avg1': 58.99, 'avg7': 65.3, 'avg30': 54.98, 'avg-holo': None, 'low-holo': None, 'trend-holo': 19.66, 'avg1-holo': 7.99, 'avg7-holo': 14.29, 'avg30-holo': 25.6}, 'tcgplayer': {'unit': 'USD', 'updated': '2026-09-30T22:54:33.913Z', 'holofoil': {'produc

In [6]:
def extract_card_data(card):
    pricing = card.get('pricing', {})
    tcgplayer = pricing.get('tcgplayer', {})
    cardmarket = pricing.get('cardmarket', {})
    
    price_usd = None
    if tcgplayer:
        # tcgplayer pricing is nested under a variant key (holofoil, normal, etc.)
        # grab the first variant dict we find and pull its marketPrice
        for key, value in tcgplayer.items():
            if isinstance(value, dict) and 'marketPrice' in value:
                price_usd = value['marketPrice']
                break
    
    if price_usd is None and cardmarket:
        price_usd = cardmarket.get('avg')
    
    return {
        'id': card.get('id'),
        'name': card.get('name'),
        'set': card.get('set', {}).get('name'),
        'rarity': card.get('rarity'),
        'illustrator': card.get('illustrator'),
        'hp': card.get('hp'),
        'types': card.get('types'),
        'price_usd': price_usd
    }

print(extract_card_data(card_detail))

{'id': 'base1-1', 'name': 'Alakazam', 'set': 'Base Set', 'rarity': 'Rare', 'illustrator': 'Ken Sugimori', 'hp': 80, 'types': ['Psychic'], 'price_usd': 69.33}


In [7]:
import time

set_ids = ['base1', 'base2', 'base3']
all_cards = []

for set_id in set_ids:
    set_response = requests.get(f"https://api.tcgdex.net/v2/en/sets/{set_id}")
    brief_cards = set_response.json()['cards']
    
    print(f"Fetching {len(brief_cards)} cards from {set_id}...")
    
    for brief_card in brief_cards:
        card_response = requests.get(f"https://api.tcgdex.net/v2/en/cards/{brief_card['id']}")
        card_detail = card_response.json()
        all_cards.append(extract_card_data(card_detail))
        time.sleep(0.1)  # small pause so we're not hammering the API

print(f"Done. Pulled {len(all_cards)} cards total.")

Fetching 102 cards from base1...
Fetching 64 cards from base2...
Fetching 62 cards from base3...
Done. Pulled 228 cards total.


In [10]:
import pandas as pd

df = pd.DataFrame(all_cards)
df.to_csv('pokemon_cards_raw.csv', index=False)

print(df.shape)
df.head()

(228, 8)


,id,name,set,rarity,illustrator,hp,types,price_usd
0,base1-1,Alakazam,Base Set,Rare,Ken Sugimori,80.0,[Psychic],69.33
1,base1-2,Blastoise,Base Set,Rare,Ken Sugimori,100.0,[Water],222.34
2,base1-3,Chansey,Base Set,Rare,Ken Sugimori,120.0,[Colorless],57.57
3,base1-4,Charizard,Base Set,Rare,Mitsuhiro Arita,120.0,[Fire],944.53
4,base1-5,Clefairy,Base Set,Rare,Ken Sugimori,40.0,[Colorless],52.79
